In [2]:
print(1)

1


In [3]:
import sys
from pathlib import Path

# Поднимаемся к корневой папке проекта  
# (где лежит avsr_conformer/)
PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in dir() else Path.cwd()
PROJECT_ROOT

WindowsPath('c:/Users/Tim/Desktop/UNIVERSITY/ВКР/avsr_conformer/notebooks')

In [4]:
sys.path.insert(0, str(PROJECT_ROOT))

In [6]:
import os

print("Текущая:", os.getcwd())
print("\nНа уровень выше:", os.path.abspath(".."))
print("\nЧто на уровень выше:")
for item in os.listdir(".."):
    print(f"  {item}")

print("\nЕсть ли там avsr_conformer/avsr?")
print("  ", os.path.exists("../avsr_conformer/avsr"))
print("\nЕсть ли там avsr_conformer/inference?")
print("  ", os.path.exists("../avsr_conformer/inference"))
print("\nЕсть ли просто inference?")
print("  ", os.path.exists("../inference"))

Текущая: c:\Users\Tim\Desktop\UNIVERSITY\ВКР\avsr_conformer\notebooks

На уровень выше: c:\Users\Tim\Desktop\UNIVERSITY\ВКР\avsr_conformer

Что на уровень выше:
  avsr
  checkpoints
  configs
  notebooks
  README.md
  requirements.txt
  scripts
  setup.py
  utils

Есть ли там avsr_conformer/avsr?
   False

Есть ли там avsr_conformer/inference?
   False

Есть ли просто inference?
   False


In [2]:
import sys
sys.path.insert(0, "..")   # из notebooks/ поднимаемся в avsr_conformer/

# Импорт начинается с avsr, без avsr_conformer
from avsr.inference.transcribe import transcribe_video

Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
No known unigrams provided, decoding results might be a lot worse.


In [ ]:
# 1
from utils.checkpoints import load_model

model = load_model("../checkpoints/conformer_specaug_5.pt")
video_path = "../../test_videos/0511.mp4"

result = transcribe_video(
        video_path=video_path,
        model=model,
        segment_sec=20.0,
        use_video=False, 
        reference_text=None,      # передай строку для подсчёта WER
    )

Loaded checkpoint: epoch=3, WER=0.576

Transcribing: ../../test_videos/0511.mp4
  [1/3] Extracting audio...
  [2/3] Splitting into segments...
12 segments found
  [3/3] Transcribing...
    [00:00:00 – 00:00:20]  всем приветнязават бы реструшенные, я учительмы тематикеь от, и мы сламяюм сегодня не ношку быболтаем т вы такой стри меня очем стрим пропоболтать, но, там е заявленые тема, что-то промотиматику, зачем нужна отемати ки как-люб
    [00:00:19 – 00:00:39]  потсемати, ка как е полюбить и, давайте начнем с этуг одальше я буду стараться отведть на, каки-то, ваше вопросаем, да, все, что хотите, можете спрашивать, и она это потвечаю вот, если увижу ваше вопросо, смотрити, а, что ако
    [00:00:38 – 00:00:58]  чтотакое маэтематикгда, и как е полюбить, точне рош ворос о нам жто ян не наит не отвечать,я, как яобще прибил атематигда, как этой умногих полычается ковота, в девствия хорошо о получается быструбериси и, они станоется с орот сменыви по-берную куково пы хороше
    [00:00:57 – 00:

In [13]:
from pyctcdecode import build_ctcdecoder

from avsr.data.tokenizer import tokenizer

# tokenizer.id2char это dict {0: "<blank>", 1: "<unk>", 2: "а", ...}
# pyctcdecode хочет список где blank в позиции 0
labels = [tokenizer.id2char[i] for i in range(len(tokenizer.id2char))]

# Заменяем спец-токены на пустую строку (для blank pyctcdecode так и ожидает)
labels[0] = ""   # blank
# unk и остальные оставляем как есть

# beam_decoder = build_ctcdecoder(
#     labels=labels,
#     kenlm_model_path=None,   # пока без LM
# )
beam_decoder_1 = build_ctcdecoder(
    labels=labels,
    # kenlm_model_path="C:/kenlm_models/ru_4gram.bin",   # пока без LM
)

In [14]:
# без kenLM
from utils.checkpoints import load_model

model = load_model("../checkpoints/conformer_specaug_5.pt")
video_path = "../../test_videos/0511.mp4"

result = transcribe_video(
        video_path=video_path,
        model=model,
        segment_sec=20.0,
        use_video=False, 
        reference_text=None,      # передай строку для подсчёта WER
        decoder=beam_decoder_1
    )


Loaded checkpoint: epoch=3, WER=0.576

Transcribing: ../../test_videos/0511.mp4
  [1/3] Extracting audio...
  [2/3] Splitting into segments...
12 segments found
  [3/3] Transcribing...
    [00:00:00 – 00:00:20]  всем приветнязават бы реструшенные, я учительмы тематикеь от, и мы сламяюм сегодня не ножку быболтаем т вы такой стри меня очем стрим пропоболтать, но, там е заявленые тема, что-то промотиматику, зачем нужна отемати ки как-люб
    [00:00:19 – 00:00:39]  потсемати, ка как е полюбить и, давайте начнем с этуг одальше я буду стараться отведть на, каки-то, ваше вопросаем, да, все, что хотите, можете спрашивать, и она это потвечаю вот, если увижу ваше вопросо, смотрити, а, что ако
    [00:00:38 – 00:00:58]  чтотакое маэтематикгда, и как е полюбить, точне рош ворос о нам жто ян не наит не отвечать,я, как яобще прибил атематигда, как этой умногих полычается уковота, в девствия хорошо о получается быструбериси и, они станоется с орот сменыви по-берную куково пы хороше
    [00:00:57 – 00

In [15]:
# с kenLM
from utils.checkpoints import load_model

model = load_model("../checkpoints/conformer_specaug_5.pt")
video_path = "../../test_videos/0511.mp4"

result = transcribe_video(
        video_path=video_path,
        model=model,
        segment_sec=20.0,
        use_video=False, 
        reference_text=None,      # передай строку для подсчёта WER
    )


Loaded checkpoint: epoch=3, WER=0.576

Transcribing: ../../test_videos/0511.mp4
  [1/3] Extracting audio...
  [2/3] Splitting into segments...
12 segments found
  [3/3] Transcribing...
    [00:00:00 – 00:00:20]  всем привет нязавобыреструшенны я учитель тематике от и мы сламюм,ясегодня не ножку болтаем то вы такой стри меня чем стрим про поболтать но там заявлено тема что-то, промотиматику, зачем нужна тематики как люб
    [00:00:19 – 00:00:39]  потсемати,ка как е полюбить и давайте начнем с это дальше я буду стараться ответ на каки-то, ваше вопросам да всё что хотите можете спрашивать на это отвечаю вот если увижу ваш вопрос смотрите что ко
    [00:00:38 – 00:00:58]  что такое математика да как полюбить очень рошеворостонам то не на не отвечать как я общей прибил атематигда, как это многих получается коота действия хорошо получается быструбериии, ни таноетсясорот мены побернуюкуково по хорошо
    [00:00:57 – 00:01:17]  куково по хорошо получаются не зная вышивать крестик моне тоже он 

In [11]:
# 1.2
from utils.checkpoints import load_model

model = load_model("../checkpoints/conformer_specaug_5.pt")
video_path = "../../test_videos/0511.mp4"

result = transcribe_video(
        video_path=video_path,
        model=model,
        segment_sec=20.0,
        use_video=False, 
        reference_text=None,      # передай строку для подсчёта WER
    )

Loaded checkpoint: epoch=3, WER=0.576

Transcribing: ../../test_videos/0511.mp4
  [1/3] Extracting audio...
  [2/3] Splitting into segments...
12 segments found
  [3/3] Transcribing...
    [00:00:00 – 00:00:20]  всем привет нязавобыреструшенны я учитель тематике от и мы сламюм,ясегодня не ножку болтаем то вы такой стри меня чем стрим про поболтать но там заявлено тема что-то, промотиматику, зачем нужна тематики как люб
    [00:00:19 – 00:00:39]  потсемати,ка как е полюбить и давайте начнем с это дальше я буду стараться ответ на каки-то, ваше вопросам да всё что хотите можете спрашивать на это отвечаю вот если увижу ваш вопрос смотрите что ко
    [00:00:38 – 00:00:58]  что такое математика да как полюбить очень рошеворостонам то не на не отвечать как я общей прибил атематигда, как это многих получается коота действия хорошо получается быструбериии, ни таноетсясорот мены побернуюкуково по хорошо
    [00:00:57 – 00:01:17]  куково по хорошо получаются не зная вышивать крестик моне тоже он 

In [3]:
# 1.3
from utils.checkpoints import load_model

model = load_model("../checkpoints/conformer_specaug_6_kenlm.pt")
video_path = "../../test_videos/0511.mp4"

result = transcribe_video(
        video_path=video_path,
        model=model,
        segment_sec=20.0,
        use_video=False, 
        reference_text=None,      # передай строку для подсчёта WER
    )

Loaded checkpoint: epoch=5, WER=0.537

Transcribing: ../../test_videos/0511.mp4
  [1/3] Extracting audio...
  [2/3] Splitting into segments...
12 segments found
  [3/3] Transcribing...
    [00:00:00 – 00:00:20]  всем прветнзавтбыреструшенны я учитель тематике от и мы сламет сегодня не ножку болтаем т вы такой стри меня чем стрим про поболтать но там заявлено тема чото промотиматику, зачем нужна тематики как лю
    [00:00:19 – 00:00:39]  потематикактъе полюбить и давайте начнем с это одалиьше, я буду стараться ответить на такие-то, ваше вопросам да всё что хотите можете спрашивать на это отвечаю вот если увижу ваш вопрос смотрите что так
    [00:00:38 – 00:00:58]  что такое математик да как полюбить очерошеворосто ам то не знает не отвечать я как я общей прибил натиматегда, как это многих улучает ся коота действия хорошо получается быструбериции, ни тноетсяспротсменыи победную куково пы хорошо
    [00:00:57 – 00:01:17]  куково ты хорошо получаются не зная вышивать крестик моне тоже он о

In [ ]:
# from avsr.inference.transcribe import transcribe_video
from avsr.inference.decoder import beam_decoder

ModuleNotFoundError: No module named 'avsr_inference'

In [9]:
# без kenLM
from utils.checkpoints import load_model

model = load_model("../checkpoints/conformer_specaug_5.pt")
video_path = "../../test_videos/savateev.mp4"

result = transcribe_video(
        video_path=video_path,
        model=model,
        segment_sec=20.0,
        use_video=False, 
        reference_text=None,      # передай строку для подсчёта WER
        length_secs=60*3.5,
        decoder=beam_decoder_1
    )


Loaded checkpoint: epoch=3, WER=0.576

Transcribing: ../../test_videos/savateev.mp4
  [1/3] Extracting audio...
  [2/3] Splitting into segments...
50 segments found
Обрабатываем первые 210.0 секунд
После ограничения: 12
  [3/3] Transcribing...
    [00:00:00 – 00:00:20]  давайте детальна, расмотрим ся жет, ⁇ связанныис стем носкоь ода леков видно горы⁇пи так⁇ это земля⁇ вемноша.
    [00:00:19 – 00:00:39]  но ша⁇ вых⁇ нахоедесь в этой точке, на претом выо зобрались на⁇ нектуру горувы ходите узнать настоко насклко дали посньевидно, что я т означает ⁇ это зачайствго провоитеа.
    [00:00:38 – 00:00:58]  чес того провоите а доточни касательна в этойгвэту сторонутось, вопросносколо ковидна, незаесто того в какую сторону, высмотрите п этого можно обще, считать, что за дача плоскоя, в данном случая, не важно, что земля являются шаромможет
    [00:00:57 – 00:01:17]  можно провестие, плоскость, черес центор землииточку, каторойвы наховате с направления, котором восмотрите, по учто такая, соверше

In [ ]:
# с kenLM
from utils.checkpoints import load_model

model = load_model("../checkpoints/conformer_specaug_5.pt")
video_path = "../../test_videos/savateev.mp4"

result = transcribe_video(
        video_path=video_path,
        model=model,
        segment_sec=20.0,
        use_video=False, 
        reference_text=None,      # передай строку для подсчёта WER
        length_secs=60*3.5,
    )

Loaded checkpoint: epoch=3, WER=0.576

Transcribing: ../../test_videos/savateev.mp4
  [1/3] Extracting audio...
  [2/3] Splitting into segments...
50 segments found
Обрабатываем первые 210.0 секунд
После ограничения: 12
  [3/3] Transcribing...
    [00:00:00 – 00:00:20]  давайте детально рассмотрим ся жет связаны сем носкокодалековидно горы⁇питак⁇ это земля вемноша.
    [00:00:19 – 00:00:39]  но шар вых нахоесьвэтойточке, на претом во забрались на некоторую гору вы ходите узнать ностока насклкодалипоснье ино что я означает это зачайствопровоитеа.
    [00:00:38 – 00:00:58]  чество проводит за доточнийкасательн этой этусторнтось, вопросносколоковидна незаестотого,в какую сторону вы смотрите по этого можно обе считать что задача плоская в данном случае не важно что земля является шаром может
    [00:00:57 – 00:01:17]  можно провести плоскость через центр земли точку которой ынаховатесе направления котором во смотрите по учета такая совершенно деме соверена полним плань
    [00:01:16 – 00:0

In [2]:
# 2
from utils.checkpoints import load_model

model = load_model("../checkpoints/conformer_specaug_5.pt")
video_path = "../../test_videos/savateev.mp4"

result = transcribe_video(
        video_path=video_path,
        model=model,
        segment_sec=20.0,
        use_video=False, 
        reference_text=None,      # передай строку для подсчёта WER
        length_secs=60*3.5
    )

Loaded checkpoint: epoch=3, WER=0.576

Transcribing: ../../test_videos/savateev.mp4
  [1/3] Extracting audio...
  [2/3] Splitting into segments...
50 segments found
Обрабатываем первые 210.0 секунд
После ограничения: 12
  [3/3] Transcribing...
    [00:00:00 – 00:00:20]  давайте детально рассмотрим ся жет связаны сем носкокодалековидно горы⁇питак⁇ это земля вемноша.
    [00:00:19 – 00:00:39]  но шар вых нахоесьвэтойточке, на претом во забрались на некоторую гору вы ходите узнать ностока насклкодалипоснье ино что я означает это зачайствопровоитеа.
    [00:00:38 – 00:00:58]  чество проводит за доточнийкасательн этой этусторнтось, вопросносколоковидна незаестотого,в какую сторону вы смотрите по этого можно обе считать что задача плоская в данном случае не важно что земля является шаром может
    [00:00:57 – 00:01:17]  можно провести плоскость через центр земли точку которой ынаховатесе направления котором во смотрите по учета такая совершенно деме соверена полним плань
    [00:01:16 – 00:0

In [4]:
# 2.1 kenlm
from utils.checkpoints import load_model

model = load_model("../checkpoints/conformer_specaug_6_kenlm.pt")
video_path = "../../test_videos/savateev.mp4"

result = transcribe_video(
        video_path=video_path,
        model=model,
        segment_sec=20.0,
        use_video=False, 
        reference_text=None,      # передай строку для подсчёта WER
        length_secs=60*3.5
    )

Loaded checkpoint: epoch=5, WER=0.537

Transcribing: ../../test_videos/savateev.mp4
  [1/3] Extracting audio...
  [2/3] Splitting into segments...
50 segments found
Обрабатываем первые 210.0 секунд
После ограничения: 12
  [3/3] Transcribing...
    [00:00:00 – 00:00:20]  давайте детально рассмотрим саужет связанные семноскодалековидно горы и так это земля земноша.
    [00:00:19 – 00:00:39]  но шавых⁇нахоедесь этой очки на притом во забрались на некоторую гору вы ходите узнать носток наскльодалипосне идна что означает это зачайствопровоите
    [00:00:38 – 00:00:58]  чество проводит за доточнийкасательв этой это сторонтось, вопросно скогоковидна, незаестотого какую сторону вы смотрите по этого можно обе считать что задача плоская данном случае не важно что земля является шаромможет
    [00:00:57 – 00:01:17]  можно провести плоскость через центр земли точку которой ы напоите направления котором во смотрите по учета такая совершенно деме соверена полним плань
    [00:01:16 – 00:01:36]  онем

In [2]:
# 2
from utils.checkpoints import load_model

model = load_model("../checkpoints/conformer_sova_1.pt")
video_path = "../../test_videos/savateev.mp4"

result = transcribe_video(
        video_path=video_path,
        model=model,
        segment_sec=20.0,
        use_video=False, 
        reference_text=None,      # передай строку для подсчёта WER
        length_secs=60*3.5
    )

  Новые параметры (инициализированы случайно): 58
Loaded checkpoint: epoch=8, WER=0.882

Transcribing: ../../test_videos/savateev.mp4
  [1/3] Extracting audio...
  [2/3] Splitting into segments...
50 segments found
Обрабатываем первые 210.0 секунд
После ограничения: 12
  [3/3] Transcribing...
    [00:00:00 – 00:00:20]  давайти деталин расмотрим сужет свезнны с слем носклко д ликови назгорыйи такэто зимля земныша
    [00:00:19 – 00:00:39]  ноше вы нахойдесь в эт точки но претомо забралис на нектрыю горул выхотител знае настоко насокода н посне нидно что я то значает это зачаечтого прободтиза
    [00:00:38 – 00:00:58]  есто прогод те н оточни касательно в это вэто сторно тось о просно склько вино незвст тоговкакую сторну высмотрит п это можно вобще читать что задаче плоскоя в да ном слочеря не важно что зим ле влят сяшаром может
    [00:00:57 – 00:01:17]  можно провестия плоскасть челесценто зимыточку кто ы нахо тесенапровления которо госмотрете получ что така сроше на дем среше наплонем